# Matirx Multiplication in Triton

这里编写一个高性能FP16矩阵乘法内核，性能可以和 cuBLAS 或 rocBLAS 媲美。

要学习的内容
- 块级矩阵乘法 Block-level matrix multiplications.
- 多维指针运算 Multi-dimensional pointer arithmetic.
- 程序重排序以提高 L2 缓存命中率 Program re-ordering for improved L2 cache hit rate.
- 自动化性能调优 Automatic performance tuning.


## Motivations

矩阵乘法是大模型关键模块，硬件内部的内核库例如cuBLAS无法定制（融合激活函数），这里学习使用 Triton 实现矩阵乘。

编写的内核将实现以下分块算法，用于将（M, K）矩阵与（K，N）矩阵相乘，伪代码如下：

In [ ]:
# Do in parallel
for m in range(0, M, BLOCK_SIZE_M):
    # Do in parallel
    for n in range(0, N, BLOCK_SIZE_N):
        acc = zeros((BLOCK_SIZE_M, BLOCK_SIZE_N)):
        for k in range(0, K, BLOCK_SIZE_K):
            a = A[m : m + BLOCK_SIZE_M, k : k + BLOCK_SIZE_K]
            b = B[k : k + BLOCK_SIZE_K, n : n + BLOCK_SIZE_N]
            acc += dot(a, b)
        C[m : m + BLOCK_SIZE_M, n : n + BLOCK_SIZE_N] = acc


其中，双重嵌套 for 循环的每次迭代都由一个专用 Triton 程序实例执行。

## Compute Kernel

Triton中实现很简单，难点在于内层循环中必须读取A和B块的内存位置。因此需要多维指针运算。

### Pointer Arithmetic 指针运算

对于行主序的二维张量 X ， X[i, j] 的内存位置由 &X[i, j] = X + i*stride_xi + j*stride_xj 给出。因此， A[m : m+BLOCK_SIZE_M, k:k+BLOCK_SIZE_K] 和 B[k : k+BLOCK_SIZE_K, n : n+BLOCK_SIZE_N] 的指针块可以用伪代码定义为：

In [ ]:
&A[m : m+BLOCK_SIZE_M, k:k+BLOCK_SIZE_K] =  a_ptr + (m : m+BLOCK_SIZE_M)[:, None]*A.stride(0) + (k : k+BLOCK_SIZE_K)[None, :]*A.stride(1);
&B[k : k+BLOCK_SIZE_K, n:n+BLOCK_SIZE_N] =  b_ptr + (k : k+BLOCK_SIZE_K)[:, None]*B.stride(0) + (n : n+BLOCK_SIZE_N)[None, :]*B.stride(1);